# 04. LifeMate AI Agent — Huấn Luyện QLoRA & Tạo Bài Nộp trên Kaggle (GPU T4 x2 / P100)

> **Mục tiêu:** Fine-tune mô hình **Qwen3-4B-Instruct-2507** trên 5.086 lượt kịch bản hội thoại chuẩn ChatML của Zalo AI Challenge 2026.
> **Thời gian huấn luyện:** ~15 - 25 phút (thay vì bị nghẽn 12 tiếng do cấu hình đa GPU ảo hóa).
> **Kết quả đầu ra:** Adapter LoRA tối ưu + File nộp bài `submission_public_test.jsonl` chuẩn hợp lệ 7 tầng.


## 1. Kiểm tra Card Đồ Họa (GPU P100 Status)

In [ ]:
!nvidia-smi

## 2. Cài đặt các thư viện AI cần thiết

In [ ]:
!pip install -q "transformers>=4.45" accelerate peft bitsandbytes datasets sentencepiece huggingface_hub

## 3. Clone Mã Nguồn vào `/kaggle/working/LifeMateAgent`

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/kaggle/working/LifeMateAgent")
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git {WORKSPACE}

%cd {WORKSPACE}
!git checkout main
!git pull origin main

## 4. Chuẩn bị Dữ Liệu Train & Test
Hệ thống sẽ tự động quét thư mục `/kaggle/input/` để phát hiện dữ liệu (kể cả đã giải nén hay file `.zip`).

In [ ]:
import os, shutil
from pathlib import Path

DATA_DIR = WORKSPACE / "train+public_test"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Tìm và liên kết thư mục train/ và public_test/ nếu Kaggle đã giải nén sẵn
if Path("/kaggle/input").exists():
    found_train = list(Path("/kaggle/input").rglob("train.jsonl"))
    found_test = list(Path("/kaggle/input").rglob("public_test.jsonl"))

    if found_train and not (DATA_DIR / "train" / "train.jsonl").exists():
        train_src = found_train[0].parent
        target_train = DATA_DIR / "train"
        try:
            os.symlink(str(train_src), str(target_train))
            print(f"🔗 Đã liên kết thư mục train từ: {train_src}")
        except Exception:
            !cp -r "{train_src}" "{target_train}"

    if found_test and not (DATA_DIR / "public_test" / "public_test.jsonl").exists():
        test_src = found_test[0].parent
        target_test = DATA_DIR / "public_test"
        try:
            os.symlink(str(test_src), str(target_test))
            print(f"🔗 Đã liên kết thư mục public_test từ: {test_src}")
        except Exception:
            !cp -r "{test_src}" "{target_test}"

# 2. Nếu dữ liệu dạng file zip trong /kaggle/input, tự động giải nén
if not (DATA_DIR / "train" / "train.jsonl").exists() and Path("/kaggle/input").exists():
    zips = list(Path("/kaggle/input").rglob("*.zip"))
    for z in zips:
        if "train" in z.name.lower() or "lifemate" in z.name.lower() or "test" in z.name.lower():
            print(f"📦 Đang giải nén zip từ: {z}")
            !unzip -q -o "{z}" -d {WORKSPACE}/
            break

# Kiểm tra tính sẵn sàng
train_ok = (DATA_DIR / "train" / "train.jsonl").exists()
test_ok = (DATA_DIR / "public_test" / "public_test.jsonl").exists()

if train_ok and test_ok:
    print("\n✅ Dữ liệu train (1.545 kịch bản) và public_test (634 mẫu) đã sẵn sàng 100%!")
elif train_ok:
    print("\n✅ Dữ liệu train (1.545 kịch bản) đã sẵn sàng!")
else:
    print("\n⚠️ Chưa thấy train.jsonl trong /kaggle/input!")
    print("👉 Cách nạp: Bấm '+ Add Input' ở góc trên bên phải Kaggle để upload file train+public_test.zip")


## 5. Bóc tách 1.545 kịch bản thành 5.086 lượt huấn luyện SFT (ChatML)
Mỗi lượt huấn luyện mô phỏng Teacher-Forcing có mặt đầy đủ các thông tin ngữ cảnh, danh mục tool và phản hồi trợ lý.

In [ ]:
!python scripts/prepare_sft_data.py

## 6. Huấn luyện LoRA trên Kaggle GPU (Thời gian: ~15 - 25 phút)
> ⚠️ **Giải mã nguyên nhân nghẽn 72s/step khi dùng GPU T4 x2 trên Kaggle:**
> - Kaggle cấp 2 card T4 (mỗi card 15 GB VRAM). Tuy nhiên, môi trường ảo hóa của Kaggle **tắt tính năng Peer-to-Peer (P2P)** qua PCIe giữa 2 GPU.
> - Khi chia nửa mô hình sang GPU 0 và nửa sang GPU 1 (`device_map='auto'`), toàn bộ dữ liệu phải chạy vòng qua RAM CPU máy chủ hàng nghìn lần mỗi bước $ightarrow$ gây thắt cổ chai và CPU bị đơ, kéo dài tới 72 giây/bước!
> - **Giải pháp tối ưu nhất:** Mô hình Qwen3-4B lượng tử hóa 4-bit chỉ chiếm **~4.5 GB VRAM**, thừa sức nằm gọn trong **1 card T4 duy nhất (15 GB)**.
> - Bằng cách gán `CUDA_VISIBLE_DEVICES=0`, toàn bộ mô hình nằm trên GPU 0 với nhân Turing Tensor Cores siêu tốc. Tốc độ tăng vọt từ 72s/step xuống **~0.4 - 0.6s/step (nhanh gấp ~120 lần)**!
> - **Toàn bộ 3 Epochs (1.908 steps) chỉ mất khoảng ~15 - 20 phút** để hoàn thành!
> - Hệ thống tự động lưu Checkpoint định kỳ mỗi 200 steps (mỗi ~2 phút) và hỗ trợ cờ `--resume` nếu có gián đoạn mạng.

In [ ]:
ADAPTER_OUT = "/kaggle/working/qwen3_4b_lora"

# Chạy huấn luyện trên GPU 0 (Tối ưu tuyệt đối cho Kaggle T4 x2 & P100):
# - Tránh nghẽn PCIe/CPU giữa 2 card T4, tốc độ ~0.5s/step (xong 3 Epochs trong ~15-20 phút)
# - VRAM chỉ chiếm ~4.5 GB / 15 GB của GPU 0
# - Tự động lưu checkpoint mỗi 200 steps, hỗ trợ tự khôi phục với --resume
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora.py \
    --epochs 3 \
    --lr 2e-4 \
    --batch-size 1 \
    --grad-accum 8 \
    --save-steps 200 \
    --resume \
    --output-dir {ADAPTER_OUT}


## 7. Kiểm tra Adapter & Nén file zip cho Kaggle Output

In [ ]:
import shutil
from pathlib import Path

adapter_dir = Path(ADAPTER_OUT)
has_final = (adapter_dir / "adapter_model.safetensors").exists() or (adapter_dir / "adapter_model.bin").exists()

# Nếu chưa có bản lưu cuối nhưng có checkpoint trung gian, tự động copy checkpoint mới nhất
if not has_final and (adapter_dir / "checkpoints").exists():
    ckpts = sorted(list((adapter_dir / "checkpoints").glob("checkpoint-*")), key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else 0)
    if ckpts:
        latest = ckpts[-1]
        print(f"⚠️ Chưa có final export, đang copy checkpoint mới nhất: {latest.name}...")
        for item in latest.iterdir():
            if item.is_file():
                shutil.copy2(item, adapter_dir / item.name)
        has_final = (adapter_dir / "adapter_model.safetensors").exists() or (adapter_dir / "adapter_model.bin").exists()

if has_final:
    print(f"🎉 Huấn luyện thành công! Trọng số lưu tại: {adapter_dir}")
    # Nén zip để tải về trực tiếp từ tab Output trên thanh bên phải Kaggle
    zip_path = Path("/kaggle/working/qwen3_4b_lora_adapter")
    shutil.make_archive(str(zip_path), "zip", str(adapter_dir))
    print(f"📦 File zip tải về: {zip_path}.zip")
else:
    print("❌ Chưa tìm thấy adapter weights. Vui lòng kiểm tra lại tiến trình train.")


## 8. Đẩy LoRA Adapter lên Hugging Face Hub (Khuyên dùng - Private Repo)
Adapter LoRA chỉ nặng **~50MB**. Đưa lên Hugging Face Hub giúp lưu trữ vĩnh viễn và có thể nạp thẳng ở bất kỳ đâu.

### 🔑 Cách lấy Hugging Face Token miễn phí (30 giây):
1. Truy cập: https://huggingface.co/settings/tokens
2. Nhấp nút **Create new token** $\rightarrow$ Chọn Type: **Write** (bắt buộc chọn Write để có quyền tải model lên) $\rightarrow$ Đặt tên bất kỳ $\rightarrow$ Nhấp **Create token**.
3. Sao chép chuỗi mã bắt đầu bằng `hf_...` và dán vào biến `HF_TOKEN` ở ô code bên dưới.

In [ ]:
# ==============================================================================
# ĐIỀN THÔNG TIN TÀI KHOẢN HUGGING FACE CỦA BẠN TẠI ĐÂY:
# ==============================================================================
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <-- DÁN WRITE TOKEN CỦA BẠN VÀO ĐÂY
HF_REPO_ID = "your_username/lifemate-qwen3-4b-lora" # <-- ĐỔI 'your_username' THÀNH TÊN TÀI KHOẢN CỦA BẠN

token_arg = f"--token {HF_TOKEN}" if HF_TOKEN and not HF_TOKEN.startswith("hf_xxx") else ""

!python scripts/push_to_hub.py \
    --adapter-dir {ADAPTER_OUT} \
    --repo-id {HF_REPO_ID} \
    {token_arg}


## 9. Chạy Tạo Bài Nộp Cho 634 Mẫu Public Test
Chạy suy luận toàn bộ 634 mẫu với Não bộ LoRA mới và bộ lọc Guardrails chống lặp/chống ảo giác.

In [ ]:
SUBMISSION_OUT = "/kaggle/working/submission_public_test.jsonl"

!python scripts/generate_submission.py \
    --split public_test \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --adapter-path {ADAPTER_OUT} \
    --output {SUBMISSION_OUT} \
    --no-resume \
    --verbose

## 10. Kiểm tra Hợp Chuẩn 7 Lớp & Tải File Nộp Bài
Chạy kiểm tra linter chính thức để đảm bảo 100% hợp lệ trước khi gửi BTC.

In [ ]:
!python scripts/lint_submission.py {SUBMISSION_OUT}

print("\n" + "=" * 60)
print("🎉 CHÚC MỪNG BẠN! FILE NỘP BÀI ĐÃ SẴN SÀNG!")
print("👉 Hãy nhìn sang thanh bên phải (Kaggle sidebar) -> Mở tab 'Output'")
print("👉 Nhấp vào dấu 3 chấm cạnh file 'submission_public_test.jsonl' -> Chọn 'Download'")
print("👉 Nộp file vừa tải lên Leaderboard Zalo AI Challenge!")
print("=" * 60)